In [41]:
from langchain_core.tools import InjectedToolArg
from typing import Annotated
from langchain_core.tools import tool
import requests
import json
from langchain_core.messages import HumanMessage,AIMessage,ToolMessage,BaseMessage

In [42]:
from langchain_huggingface import ChatHuggingFace, HuggingFaceEndpoint
from dotenv import load_dotenv

load_dotenv()

llm = HuggingFaceEndpoint(
    repo_id="meta-llama/Llama-3.1-8B-Instruct",
    task="conversational"
)

model = ChatHuggingFace(llm=llm)

In [43]:
@tool
def get_conversion_factor(base_currency: str, target_currency: str) -> float:
  """
  This function fetches the currency conversion factor between a given base currency and a target currency
  """
  url = f'https://v6.exchangerate-api.com/v6/d61b36cab6dff181877e9c5c/pair/{base_currency}/{target_currency}'

  response = requests.get(url)

  return response.json()

@tool
def convert(base_currency_value: int, conversion_rate: Annotated[float, InjectedToolArg]) -> float:
  """
  given a currency conversion rate this function calculates the target currency value from a given base currency value
  """

  return base_currency_value * conversion_rate


In [44]:
get_conversion_factor.invoke({'base_currency':'USD','target_currency':'INR'})

{'result': 'success',
 'documentation': 'https://www.exchangerate-api.com/docs',
 'terms_of_use': 'https://www.exchangerate-api.com/terms',
 'time_last_update_unix': 1780012801,
 'time_last_update_utc': 'Fri, 29 May 2026 00:00:01 +0000',
 'time_next_update_unix': 1780099201,
 'time_next_update_utc': 'Sat, 30 May 2026 00:00:01 +0000',
 'base_code': 'USD',
 'target_code': 'INR',
 'conversion_rate': 95.7847}

In [45]:
modek = model.bind_tools([get_conversion_factor,convert])

In [46]:
messages = []
query = HumanMessage("10 dollars in inr")
messages.append(query)

In [47]:
rs = modek.invoke([query])
messages.append(rs)

In [48]:
messages

[HumanMessage(content='10 dollars in inr', additional_kwargs={}, response_metadata={}),
 AIMessage(content='To convert 10 US dollars (USD) to Indian rupees (INR), I will first retrieve the current conversion rate and then perform the conversion. Let me do that for you.', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 41, 'prompt_tokens': 168, 'total_tokens': 209}, 'model_name': 'meta-llama/Llama-3.1-8B-Instruct', 'system_fingerprint': None, 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019e74fd-ae3e-7bc1-b7b4-734baa76bf30-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 168, 'output_tokens': 41, 'total_tokens': 209})]

In [49]:
import json

for tool_call in rs.tool_calls:
  # execute the 1st tool and get the value of conversion rate
  if tool_call['name'] == 'get_conversion_factor':
    tool_message1 = get_conversion_factor.invoke(tool_call)
    # fetch this conversion rate
    conversion_rate = json.loads(tool_message1.content)['conversion_rate']
    # append this tool message to messages list
    messages.append(tool_message1)
  # execute the 2nd tool using the conversion rate from tool 1
  if tool_call['name'] == 'convert':
    # fetch the current arg
    tool_call['args']['conversion_rate'] = conversion_rate
    tool_message2 = convert.invoke(tool_call)
    messages.append(tool_message2)



In [50]:
messages

[HumanMessage(content='10 dollars in inr', additional_kwargs={}, response_metadata={}),
 AIMessage(content='To convert 10 US dollars (USD) to Indian rupees (INR), I will first retrieve the current conversion rate and then perform the conversion. Let me do that for you.', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 41, 'prompt_tokens': 168, 'total_tokens': 209}, 'model_name': 'meta-llama/Llama-3.1-8B-Instruct', 'system_fingerprint': None, 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019e74fd-ae3e-7bc1-b7b4-734baa76bf30-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 168, 'output_tokens': 41, 'total_tokens': 209})]

In [51]:
modek.invoke(messages)

BadRequestError: (Request ID: 77c5d5c1-5915-4a9a-9c9a-f02fbf161491)

Bad request:
{'message': 'Cannot set `add_generation_prompt` to True when the last message is from the assistant. Consider using `continue_final_message` instead.', 'type': 'BadRequestError', 'param': None, 'code': 400}

In [ ]:
fin = modek.invoke(messages).content

In [ ]:
fin

''